With a understanding that directional classification, RF with specified hyperparameters, and lags/rolling averages of spending and shares
show some signal, deep dive into the success to create a stronger model.

In [1]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("../data/raw/marketing_roi_dataset.csv")

# Ensure proper datetime + ordering
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df.fillna(df.median())

In [2]:
spend_cols = [c for c in df.columns if "spend" in c]

In [3]:
df["total_spend"] = df[spend_cols].sum(axis=1)

for col in spend_cols:
    df[f"{col}_share"] = df[col] / (df["total_spend"] + 1e-5)

In [4]:
df["spend_momentum_3"] = df["total_spend"].diff(3)
df["spend_momentum_7"] = df["total_spend"].diff(7)

In [5]:
df["conversion_rate"] = df["conversions"] / (df["site_visits"] + 1e-5)

In [6]:
lag_cols = spend_cols + ["organic_traffic", "email_sends", "site_visits", "conversions"]

for col in lag_cols:
    for lag in [1, 3, 7]:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)

In [7]:
for col in spend_cols:
    df[f"{col}_roll7"] = df[col].rolling(7).mean()

In [8]:
leakage_cols = [c for c in df.columns if "revenue" in c and c not in ["target_up", "target_big_up", "target_multi"]]

# Also remove raw date
drop_cols = leakage_cols

df_model = df.drop(columns=drop_cols, errors="ignore")

In [9]:
# Drop NA from lags/rolling
df_model = df_model.dropna().reset_index(drop=True)

In [10]:
df_features = df_model.drop(columns=["target_up", "target_big_up", "target_multi"], errors="ignore")

In [11]:
pd.set_option('display.max_columns', None)

In [12]:
print("Train size:", df_features.shape)
# Check leakage
print("\nColumns with 'revenue' still present:")
print([c for c in df_features.columns if "revenue" in c])

Train size: (993, 45)

Columns with 'revenue' still present:
[]


Created clean input data set, build target data

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/marketing_roi_dataset.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

In [14]:
# Base
df["revenue_diff"] = df["revenue"].diff()

# Momentum
df["revenue_momentum_3"] = df["revenue"].diff(3)
df["revenue_momentum_7"] = df["revenue"].diff(7)

# Rolling
df["revenue_roll7"] = df["revenue"].rolling(7).mean()

# Direction targets
df["target_up"] = (df["revenue_diff"] > 0).astype(int)

threshold = df["revenue_diff"].rolling(30).std()
df["target_big_up"] = (df["revenue_diff"] > threshold).astype(int)

# Multi-class
df["target_multi"] = pd.qcut(df["revenue_diff"], q=3, labels=[0,1,2])

In [15]:
target_cols = [
    "revenue",
    "revenue_diff",
    "revenue_momentum_3",
    "revenue_momentum_7",
    "revenue_roll7",
    "target_up",
    "target_big_up",
    "target_multi"
]

df_targets = df[["date"] + target_cols].copy()

In [16]:
def build_model_dataset(df_features, df_targets, target_name):
    
    df_model = df_features.merge(df_targets[["date", target_name]], on="date")

    # Drop NA (from lags/rolling)
    df_model = df_model.dropna().reset_index(drop=True)

    X = df_model.drop(columns=["date", target_name])
    y = df_model[target_name]

    # numeric only
    X = X.select_dtypes(include=[np.number])

    return X, y, df_model

In [17]:
def time_split(X, y, split=0.8):
    
    split_idx = int(len(X) * split)
    
    return (
        X.iloc[:split_idx],
        X.iloc[split_idx:],
        y.iloc[:split_idx],
        y.iloc[split_idx:]
    )

In [18]:
X, y, df_model = build_model_dataset(
    df_features,
    df_targets,
    target_name="target_big_up"
)

X_train, X_test, y_train, y_test = time_split(X, y)

Solidified x, various y parameters and helper functions for building time based train-test split

In [19]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

In [20]:
def optimize_threshold(y_true, y_prob):
    
    thresholds = np.linspace(0.1, 0.9, 50)
    
    best_acc = 0
    best_t = 0.5
    
    for t in thresholds:
        preds = (y_prob > t).astype(int)
        acc = accuracy_score(y_true, preds)
        
        if acc > best_acc:
            best_acc = acc
            best_t = t
    
    return best_t, best_acc

In [21]:
rf_grid = [
    {"n_estimators": 200, "max_depth": 4, "min_samples_leaf": 1},
    {"n_estimators": 500, "max_depth": 6, "min_samples_leaf": 1},
    {"n_estimators": 500, "max_depth": 8, "min_samples_leaf": 1},
    {"n_estimators": 500, "max_depth": 8, "min_samples_leaf": 5},
]

gb_grid = [
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 500, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 500, "max_depth": 5, "learning_rate": 0.05},
    {"n_estimators": 500, "max_depth": 5, "learning_rate": 0.10},
]

In [22]:
results = []

targets = ["target_up", "target_big_up"]  # add more if needed

for target in targets:
    
    # Build dataset
    X, y, df_model = build_model_dataset(df_features, df_targets, target)
    
    # Time split
    X_train, X_test, y_train, y_test = time_split(X, y)
    
    # Skip if only one class
    if y_train.nunique() < 2 or y_test.nunique() < 2:
        continue
    
    # -------------------
    # RANDOM FOREST
    # -------------------
    for params in rf_grid:
        
        rf = RandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1
        )
        
        rf.fit(X_train, y_train)
        y_prob = rf.predict_proba(X_test)[:, 1]
        y_pred = rf.predict(X_test)
        
        auc = roc_auc_score(y_test, y_prob)
        acc = accuracy_score(y_test, y_pred)
        
        # Threshold optimization
        best_t, best_acc = optimize_threshold(y_test, y_prob)
        
        results.append({
            "model": "RF",
            "target": target,
            **params,
            "AUC": auc,
            "Accuracy_default": acc,
            "Accuracy_optimized": best_acc,
            "Best_threshold": best_t
        })
    
    # -------------------
    # GRADIENT BOOSTING
    # -------------------
    for params in gb_grid:
        
        gb = GradientBoostingClassifier(
            **params,
            random_state=42
        )
        
        gb.fit(X_train, y_train)
        y_prob = gb.predict_proba(X_test)[:, 1]
        y_pred = gb.predict(X_test)
        
        auc = roc_auc_score(y_test, y_prob)
        acc = accuracy_score(y_test, y_pred)
        
        # Threshold optimization
        best_t, best_acc = optimize_threshold(y_test, y_prob)
        
        results.append({
            "model": "GB",
            "target": target,
            **params,
            "AUC": auc,
            "Accuracy_default": acc,
            "Accuracy_optimized": best_acc,
            "Best_threshold": best_t
        })

In [23]:
results_df = pd.DataFrame(results)

# Sort by best signal
results_df = results_df.sort_values(
    by=["AUC", "Accuracy_optimized"],
    ascending=False
)

results_df

,model,target,n_estimators,max_depth,min_samples_leaf,AUC,Accuracy_default,Accuracy_optimized,Best_threshold,learning_rate
1,RF,target_up,500,6,1.0,0.594261,0.527638,0.603015,0.442857,NaN
3,RF,target_up,500,8,5.0,0.584462,0.532663,0.592965,0.426531,NaN
2,RF,target_up,500,8,1.0,0.580420,0.547739,0.562814,0.410204,NaN
0,RF,target_up,200,4,1.0,0.577389,0.557789,0.557789,0.442857,NaN
4,GB,target_up,200,3,NaN,0.571328,0.567839,0.603015,0.459184,0.05
7,GB,target_up,500,5,NaN,0.568095,0.557789,0.567839,0.540816,0.10
6,GB,target_up,500,5,NaN,0.561123,0.547739,0.562814,0.508163,0.05
5,GB,target_up,500,3,NaN,0.553445,0.562814,0.572864,0.508163,0.05
12,GB,target_big_up,200,3,NaN,0.472584,0.819095,0.849246,0.883673,0.05
13,GB,target_big_up,500,3,NaN,0.462130,0.809045,0.849246,0.900000,0.05


In [25]:
best_row = results_df.iloc[0]

best_params_raw = results_df.iloc[0].to_dict()

# Clean + cast parameters
best_params = {
    "n_estimators": int(best_params_raw["n_estimators"]),
    "max_depth": int(best_params_raw["max_depth"]) if pd.notnull(best_params_raw["max_depth"]) else None,
    "min_samples_leaf": int(best_params_raw["min_samples_leaf"]),
}

X, y, _ = build_model_dataset(df_features, df_targets, best_row["target"])
X_train, X_test, y_train, y_test = time_split(X, y)

rf_best = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

rf_best.fit(X_train, y_train)

importances = pd.Series(
    rf_best.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances.head(20)

google_ads_spend_lag1         0.056937
spend_momentum_3              0.041527
spend_momentum_7              0.036891
facebook_ads_spend_lag1       0.034368
total_spend                   0.032972
influencer_spend_share        0.028680
facebook_ads_spend            0.028451
influencer_spend_lag3         0.027977
influencer_spend_lag7         0.026645
facebook_ads_spend_lag7       0.025692
influencer_spend              0.025355
google_ads_spend              0.024385
email_sends_lag3              0.023576
site_visits_lag1              0.023310
organic_traffic               0.023186
conversions_lag3              0.022597
google_ads_spend_lag3         0.021834
email_sends                   0.021568
email_marketing_spend_lag1    0.021497
site_visits_lag7              0.021199
dtype: float64